https://github.com/ventolab/CellphoneDB/blob/master/notebooks/0_prepare_your_data_from_anndata.ipynb

## Import modules

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import os
import sys
from scipy import sparse
import pickle
import matplotlib.pyplot as plt
import kktk

sc.settings.verbosity = 1  # verbosity: errors (0), warnings (1), info (2), hints (3)

/rfs/project/rfs-iCNyzSAaucw/kk837/github/kktk/kktk/plotting.py:34: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if sc.__version__.startswith("1.4"):


In [2]:
import session_info
session_info.show()

/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


## Set path

In [3]:
data_object_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025'
path_adata_tacco = f'{data_object_dir}/TACCO/tacco-splitted_ref-HD-SCN_filtered_rm-unclassified_raw.h5ad'
path_adata_orginal = f'{data_object_dir}/all_b2c_cells_filtered_raw.h5ad'

cellphone_input_dir = f'{data_object_dir}/cellphonedb/inputs'
os.makedirs(cellphone_input_dir,exist_ok=True)

In [4]:
celltype_col = 'fine_grain_2Feb2026'

In [7]:
niche_label_col = 'cellular_niche_23Feb2026'
niche_of_interest = ['Sinus horn','SAnode - Head','SAnode - Tail','Atrium - Right']
niche_of_interest_name = 'Sinushorn-SAnode-AtriumR'

## Read in anndata

In [8]:
# read in 
adata = sc.read_h5ad(path_adata_tacco)
adata

AnnData object with n_obs × n_vars = 189948 × 18077
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2', 'coarse_grain_tacco_ref-hd-scn', 'original_cellid_celltype', 'tacco_max_compscore', 'fine_grain_2Feb2026', 'mid_grain_2Feb2026'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_grain_tacco_ref-hd-scn_colors', 'donor

In [9]:
set(adata.obs[celltype_col])

{'ArterialEndothelialCells',
 'AtrialCardiomyocytesLeft',
 'AtrialCardiomyocytesNeurotropic',
 'AtrialCardiomyocytesRight',
 'AtrialCardiomyocytesTransitional',
 'AtrioventricularNodePacemakerCells',
 'CoronaryCapillaryEndothelialCells',
 'CoronaryMuralCells',
 'DuctusArteriosusSmoothMuscleCells',
 'EndocardialCells',
 'EpicardiumDerivedCells',
 'Fibroblasts',
 'FibroblastsCOL2A1pos',
 'FibroblastsPITX2pos',
 'GreatVesselSmoothMuscleCells',
 'InnateLymphoidCells',
 'LymphaticEndothelialCells',
 'MacrophagesCX3CR1pos',
 'MacrophagesLYVE1pos',
 'MesothelialEpicardialCells',
 'Monocytes',
 'NaturalKillerCells',
 'ParasympatheticNeurons',
 'SchwannCells',
 'SinoatrialNodePacemakerCellsHead',
 'SinoatrialNodePacemakerCellsTail',
 'SinusHornPacemakerCells',
 'SympatheticNeurons',
 'ValveEndothelialCells',
 'ValveInterstitialCells',
 'VenousEndothelialCells',
 'VentricularCardiomyocytesCompact',
 'VentricularCardiomyocytesTrabeculated',
 'VentricularConductionSystemDistal',
 'VentricularCondu

In [10]:
adata.X.data[:5]

array([1., 2., 2., 1., 1.])

In [11]:
# add short version of the cell type labels
df = pd.read_csv('/rfs/project/rfs-iCNyzSAaucw/kk837/notebooks/Foetal/finegrain_name_mapping.csv')
celltype_mapping = df.set_index('Full_name')['Short_name_finalised'].to_dict()
adata.obs.replace({celltype_col:celltype_mapping},inplace=True)
set(adata.obs[celltype_col])

/tmp/ipykernel_2092744/979726308.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs.replace({celltype_col:celltype_mapping},inplace=True)


{'AVNPC',
 'ArtEC',
 'CorCapEC',
 'CorMural',
 'DASMC',
 'EPDC',
 'Endocard',
 'FB',
 'FBCOL2A1',
 'FBPITX2',
 'GVSMC',
 'ILC',
 'LymphEC',
 'MPLYVE1',
 'MacCX3',
 'MesEpiC',
 'Mono',
 'NK',
 'ParaN',
 'SANPCHd',
 'SANPCTl',
 'SHPC',
 'SchwC',
 'SympN',
 'VCSDist',
 'VCSProx',
 'ValveEC',
 'ValveIC',
 'VenEC',
 'aCML',
 'aCMNeur',
 'aCMR',
 'aCMTrans',
 'vCMC',
 'vCMT'}

## Subset cellular niche of interest

In [12]:
# add cellular niche results
ad = sc.read_h5ad(path_adata_orginal,backed='r')
adata.obs[niche_label_col] = ad.obs[niche_label_col].reindex(adata.obs_names)
adata.obs[niche_label_col].value_counts()

cellular_niche_23Feb2026
Ventricle - Compact           58999
VCS - Proximal                17628
Ventricle - Trabeculated      14154
Valves                        13821
Epicardium - Immune           11649
Fibroblast - Immune           11195
VCS - Distal                   8841
Atrium - Right                 7716
AVnode                         6872
Coronary vessel                6822
Atrium - Left                  6002
Great vessel                   5282
Epicardium - Myocardium        4478
Ductus arteriosus              3972
Endocardium - Ventricular      3252
COL2A1-high fibroblast ECM     2432
Sinus horn                     2306
SAnode - Head                  1750
SAnode - Tail                  1535
Atrium - Transitional          1224
Erythrocytes                     18
Name: count, dtype: int64

In [13]:
adata_sub = adata[adata.obs[niche_label_col].isin(niche_of_interest)]
adata_sub

View of AnnData object with n_obs × n_vars = 13307 × 18077
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2', 'coarse_grain_tacco_ref-hd-scn', 'original_cellid_celltype', 'tacco_max_compscore', 'fine_grain_2Feb2026', 'mid_grain_2Feb2026', 'cellular_niche_23Feb2026'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_gra

## Select cell types

- Select significantly enriched cell types: refere to no07_annotate_cc-clusters, notebook_no07, odds ratio

In [14]:
celltype_sel = [
    'SHPC',
    'SANPCHd',
    'SANPCTl',
    'ParaN',
    'MPLYVE1',
    'MacCX3',
    'aCMTrans',
    'VenEC',
    'FB',
    'ILC',
    'CorCapEC',
    'LymphEC',
    'EPDC',
    'MesEpiC',
    'aCMNeur',
    'aCMR',
]
# Erythrocytes, aCML, and AVNPC were excluded, as these populations likely reflect contamination or share overlapping gene expression profiles.

In [15]:
adata_sub = adata_sub[adata_sub.obs[celltype_col].isin(celltype_sel)]
adata_sub.obs[celltype_col].values.describe()

,counts,freqs
categories,,
aCMNeur,10,0.000876
aCMR,6548,0.573480
aCMTrans,314,0.027500
CorCapEC,24,0.002102
EPDC,15,0.001314
FB,706,0.061832
ILC,15,0.001314
LymphEC,65,0.005693
MacCX3,6,0.000525


## Filter & log-normalise

In [16]:
print(adata_sub.shape)
print(adata_sub.X.data[:10])
sc.pp.filter_genes(adata_sub, min_cells=3)
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)
print(adata_sub.shape)
print(adata_sub.X.data[:10])

(11418, 18077)
[1. 1. 1. 1. 1. 1. 1. 1. 2. 1.]


/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


(11418, 17328)
[2.22104167 2.22104167 2.22104167 2.22104167 2.22104167 2.22104167
 2.22104167 2.22104167 2.85841393 2.22104167]


## Save

In [17]:
meta = adata_sub.obs.copy()
meta['cell_id'] = meta.index.astype('str')
meta = meta[['cell_id', celltype_col]]
meta[celltype_col] = meta[celltype_col].astype('str')
meta.to_csv(f'{cellphone_input_dir}/meta_{niche_of_interest_name}.txt', index=False, sep = "\t")

adata_sub.var_names = adata_sub.var_names.astype(str)
adata_sub.obs_names = adata_sub.obs_names.astype(str)
adata_sub.write(f'{cellphone_input_dir}/log_norm_counts_{niche_of_interest_name}.h5ad')

In [18]:
f'{cellphone_input_dir}/meta_{niche_of_interest_name}.txt'

'/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025/cellphonedb/inputs/meta_Sinushorn-SAnode-AtriumR.txt'

In [19]:
f'{cellphone_input_dir}/log_norm_counts_{niche_of_interest_name}.h5ad'

'/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025/cellphonedb/inputs/log_norm_counts_Sinushorn-SAnode-AtriumR.h5ad'

In [20]:
!ls -lh {cellphone_input_dir}

total 113M
-rwxrwx---+ 1 kk837 kk837 166M Mar 31 09:08 log_norm_counts_Sinushorn-SAnode-AtriumR.h5ad
-rwxrwx---+ 1 kk837 kk837  75M Mar 30 23:44 log_norm_counts_Sinushorn-SAnode.h5ad
-rwxrwx---+ 1 kk837 kk837 426K Mar 31 09:08 meta_Sinushorn-SAnode-AtriumR.txt
-rwxrwx---+ 1 kk837 kk837 203K Mar 30 23:44 meta_Sinushorn-SAnode.txt


In [21]:
adata_sub.var.head()

,gene_ids,feature_types,genome,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells
SAMD11,ENSG00000187634,Gene Expression,GRCh38,False,13492,0.014410,0.014307,98.799092,16189.0,9.692149,600
NOC2L,ENSG00000188976,Gene Expression,GRCh38,False,16660,0.015533,0.015414,98.517112,17451.0,9.767210,955
KLHL17,ENSG00000187961,Gene Expression,GRCh38,False,3314,0.003005,0.003000,99.705024,3376.0,8.124743,190
PLEKHN1,ENSG00000187583,Gene Expression,GRCh38,False,100,0.000089,0.000089,99.991099,100.0,4.615121,3
PERM1,ENSG00000187642,Gene Expression,GRCh38,False,138,0.000123,0.000123,99.987717,138.0,4.934474,7


In [ ]:
adata_sub.obs[]